In [5]:
# ================================
# DCF Valuation Dashboard - Core
# ================================

def setup_environment():
    """Installs and imports all required libraries."""
    global yf, pd, np, widgets, display, Markdown, clear_output, plt, sns

    !pip install yfinance ipywidgets -q

    import yfinance as yf
    import pandas as pd
    import numpy as np
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    import matplotlib.pyplot as plt
    import seaborn as sns
    import warnings

    sns.set(style='whitegrid')
    warnings.filterwarnings('ignore', category=FutureWarning)
    pd.options.display.float_format = '{:,.2f}'.format

    print("Environment setup complete. All libraries are imported and ready.")


def fetch_and_display_data(ticker, data_output_widget):
    """
    Fetches, cleans, and displays data.

    For the bar charts, we drop any years where all three metrics
    (Total Revenue, Net Income, Free Cash Flow) are zero, so the
    first visible year is the first year that actually has data.
    """
    global price_data, financials, company_info

    with data_output_widget:
        clear_output(wait=True)
        try:
            stock = yf.Ticker(ticker)
            company_info = stock.info

            # Auto risk-free rate
            tnx = yf.Ticker('^TNX')
            company_info['risk_free_rate_auto'] = (
                tnx.history(period='5d')['Close'].iloc[-1] / 100
            )

            # Financial statements
            bs = stock.balance_sheet
            cf = stock.cashflow
            inc = stock.financials

            # Tax rate (auto)
            if 'Income Before Tax' in inc.index and 'Income Tax Expense' in inc.index:
                income_before_tax = inc.loc['Income Before Tax'].mean()
                tax_expense = inc.loc['Income Tax Expense'].mean()
                if pd.notna(income_before_tax) and income_before_tax > 0:
                    company_info['tax_rate_auto'] = tax_expense / income_before_tax
                else:
                    company_info['tax_rate_auto'] = 0.21
            else:
                company_info['tax_rate_auto'] = 0.21

            # Cost of debt (auto)
            if 'Interest Expense' in inc.index and 'Total Debt' in bs.index:
                interest_expense = abs(inc.loc['Interest Expense'].mean())
                total_debt_mean = bs.loc['Total Debt'].mean()
                if pd.notna(total_debt_mean) and total_debt_mean > 0:
                    company_info['cost_of_debt_auto'] = interest_expense / total_debt_mean
                else:
                    company_info['cost_of_debt_auto'] = 0.05
            else:
                company_info['cost_of_debt_auto'] = 0.05

            # Price data
            price_data = stock.history(period='5y')['Close']
            if price_data.empty:
                print(f"Error: Could not fetch price data for {ticker}.")
                return

            # Latest column from each statement
            latest_bs = bs.iloc[:, 0]
            latest_cf = cf.iloc[:, 0]
            latest_inc = inc.iloc[:, 0]

            def safe_get(series, keys, default=0):
                for key in keys:
                    if key in series and pd.notna(series[key]):
                        return series[key]
                return default

            revenue = safe_get(latest_inc, ['Total Revenue', 'Revenues'])
            net_income = safe_get(latest_inc, ['Net Income', 'Net Income Common Stockholders'])
            op_cashflow = safe_get(latest_cf, ['Total Cash From Operating Activities'])
            capex = safe_get(latest_cf, ['Capital Expenditures'])
            fcf = safe_get(latest_cf, ['Free Cash Flow'], default=(op_cashflow + capex))
            total_debt_latest = safe_get(latest_bs, ['Total Debt'])
            cash_equivalents = safe_get(
                latest_bs,
                ['Cash And Cash Equivalents',
                 'Cash Cash Equivalents And Short Term Investments']
            )
            shares_outstanding = company_info.get(
                'sharesOutstanding',
                safe_get(latest_inc, ['Basic Average Shares'])
            )

            # Combine financials for the model
            financials = pd.concat([inc.T, bs.T, cf.T], axis=1)
            financials = financials.loc[:, ~financials.columns.duplicated()]
            financials.index = pd.to_datetime(financials.index).year
            financials = financials.sort_index()
            financials = financials.fillna(0)

            # Ensure required columns exist
            for col in ['Total Revenue', 'EBIT', 'Net Income',
                        'Free Cash Flow', 'Total Debt',
                        'Cash And Cash Equivalents']:
                if col not in financials.columns:
                    financials[col] = 0

            # Summary table
            summary_data = {
                'Metric': ['Latest Revenue', 'Latest Net Income', 'Latest FCF',
                           'Total Debt', 'Cash & Equivalents', 'Shares Outstanding'],
                'Value': [revenue, net_income, fcf,
                          total_debt_latest, cash_equivalents, shares_outstanding]
            }
            display(Markdown(f"### Company Summary: {ticker.upper()}"))
            display(pd.DataFrame(summary_data).style.format({'Value': '{:,.0f}'}))

            # Price chart
            display(Markdown("---"))
            plt.figure(figsize=(12, 6))
            price_data.plot(title=f'{ticker.upper()} Historical Stock Price')
            plt.ylabel('Stock Price (USD)')
            plt.xlabel('Date')
            plt.show()

            # ===== BAR CHART LOGIC (DROP PURE-ZERO YEARS, FIX TICKS) =====
            plot_df = financials.copy()

            key_cols = ['Total Revenue', 'Net Income', 'Free Cash Flow']
            existing_cols = [c for c in key_cols if c in plot_df.columns]

            if existing_cols:
                zero_mask = (plot_df[existing_cols] == 0).all(axis=1)
                if (~zero_mask).any():
                    plot_df = plot_df.loc[~zero_mask]

            if plot_df.empty:
                print("No non-zero historical financial data available to plot.")
                return

            years = plot_df.index.astype(int).tolist()
            x_positions = np.arange(len(years))
            year_labels = [str(y) for y in years]

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            fig.suptitle(f'Historical Financials ({years[0]}–{years[-1]})',
                         fontsize=16)

            for ax, col in zip(axes, ['Total Revenue', 'Net Income', 'Free Cash Flow']):
                if col in plot_df.columns:
                    values = plot_df[col].values
                else:
                    values = np.zeros(len(plot_df))

                ax.bar(x_positions, values)
                ax.set_xticks(x_positions)
                ax.set_xticklabels(year_labels, rotation=90)
                ax.set_title(col)

                yticks = ax.get_yticks()
                ax.set_yticks(yticks)
                ax.set_yticklabels(
                    [
                        f'{int(y/1e9)}B' if abs(y) >= 1e9 else
                        f'{int(y/1e6)}M' if abs(y) >= 1e6 else
                        f'{int(y)}'
                        for y in yticks
                    ]
                )

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

        except Exception as e:
            print(f"An error occurred: {e}. Data might be missing for this ticker.")


def run_dcf_model(financials_df, assumptions):
    latest_revenue = financials_df['Total Revenue'].iloc[-1]
    if latest_revenue <= 0:
        return 0, 0, 0, pd.DataFrame()

    growth_rates = []
    if assumptions['growth_model'] == 'Multi-Stage (10yr)':
        high_growth_rate, terminal_rate = (
            assumptions['revenue_growth'],
            assumptions['terminal_growth']
        )
        high_growth_years = assumptions['high_growth_years']
        fade_years = 10 - high_growth_years
        decay = (high_growth_rate - terminal_rate) / fade_years if fade_years > 0 else 0
        growth_rates.extend([high_growth_rate] * high_growth_years)
        current_growth = high_growth_rate
        for _ in range(fade_years):
            current_growth -= decay
            growth_rates.append(current_growth)
        forecast_horizon = 10
    else:
        growth_rates.extend([assumptions['revenue_growth']] * 5)
        forecast_horizon = 5

    forecast_df = pd.DataFrame(index=range(1, forecast_horizon + 1))
    revenues = [latest_revenue * (1 + growth_rates[0])]
    for i in range(1, forecast_horizon):
        revenues.append(revenues[i-1] * (1 + growth_rates[i]))
    forecast_df['Revenue'] = revenues
    forecast_df['EBIT'] = forecast_df['Revenue'] * assumptions['ebit_margin']
    forecast_df['FCF'] = forecast_df['EBIT'] * (1 - assumptions['tax_rate'])
    forecast_df['Discount Factor'] = [(1 + assumptions['wacc'])**i
                                      for i in forecast_df.index]
    forecast_df['PV of FCF'] = forecast_df['FCF'] / forecast_df['Discount Factor']

    last_fcf = forecast_df['FCF'].iloc[-1]
    if assumptions['terminal_method'] == 'Perpetuity Growth':
        terminal_value = (
            last_fcf * (1 + assumptions['terminal_growth'])
        ) / (assumptions['wacc'] - assumptions['terminal_growth'])
    else:
        terminal_value = forecast_df['EBIT'].iloc[-1] * assumptions['exit_multiple']

    pv_terminal_value = (
        terminal_value / forecast_df['Discount Factor'].iloc[-1]
    )
    enterprise_value = forecast_df['PV of FCF'].sum() + pv_terminal_value
    net_debt = (
        financials.iloc[-1].get('Total Debt', 0)
        - financials.iloc[-1].get('Cash And Cash Equivalents', 0)
    )
    equity_value = enterprise_value - net_debt
    shares_outstanding = company_info.get('sharesOutstanding', 1) or 1

    return equity_value / shares_outstanding, enterprise_value, equity_value, forecast_df


def run_valuation_display(dcf_output_widget):
    with dcf_output_widget:
        clear_output(wait=True)
        if (terminal_method_choice.value == 'Perpetuity Growth'
                and terminal_growth_slider.value >= wacc_slider.value):
            display(Markdown(
                "<p style='color:red;'><b>Warning: WACC must be greater "
                "than Terminal Growth.</b></p>"
                "<i>Reason: Denominator (WACC - g) would be <= 0.</i>"
            ))
            return
        dcf_inputs = get_dcf_inputs()
        intrinsic_value, ev, eq_v, forecast_table = run_dcf_model(
            financials, dcf_inputs
        )
        current_price = price_data.iloc[-1]
        summary = {
            'Metric': ['Current Price', 'Intrinsic Value', 'Upside',
                       'Enterprise Value', 'Equity Value'],
            'Value': [current_price, intrinsic_value,
                      (intrinsic_value/current_price)-1, ev, eq_v]
        }
        display(Markdown("### DCF Valuation Summary"))
        display(
            pd.DataFrame(summary)
            .set_index('Metric')
            .style.format({
                'Value': (
                    lambda x: f'{x:.2%}' if abs(x) < 1 and x != 0
                    else f'${x:,.2f}'
                )
            })
        )
        display(Markdown("### Free Cash Flow Projections"))
        display(
            forecast_table.style.format({
                'Revenue': '${:,.0f}',
                'EBIT': '${:,.0f}',
                'FCF': '${:,.0f}',
                'PV of FCF': '${:,.0f}',
                'Discount Factor': '{:.4f}'
            })
        )


def run_full_analysis(analysis_output_widget):
    with analysis_output_widget:
        clear_output(wait=True)

        # Base assumptions and base intrinsic value
        base_assumptions = get_dcf_inputs()
        base_iv, _, _, _ = run_dcf_model(financials, base_assumptions)
        current_price = price_data.iloc[-1]

        # 1. Price vs intrinsic bar chart
        display(Markdown("### Valuation Comparison"))
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        bars = ax.bar(
            ['Current Price', 'Intrinsic Value'],
            [current_price, base_iv],
            color=['skyblue',
                   'lightgreen' if base_iv > current_price else 'salmon']
        )
        ax.set_ylabel('Price (USD)')
        ax.set_title('Market Price vs. Intrinsic Value')
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width()/2.0,
                bar.get_height(),
                f'${bar.get_height():.2f}',
                va='bottom',
                ha='center'
            )
        plt.show()

        # 2. Scenario Analysis: Bear vs Neutral vs Bull
        display(Markdown("### Scenario Analysis"))
        # Bear: lower growth & margin, higher WACC
        bear_assumptions = {
            **base_assumptions,
            'revenue_growth': base_assumptions['revenue_growth'] - 0.02,
            'ebit_margin': max(base_assumptions['ebit_margin'] - 0.02, 0.0),
            'wacc': base_assumptions['wacc'] + 0.01
        }
        # Bull: higher growth & margin, lower WACC (floored at 1%)
        bull_assumptions = {
            **base_assumptions,
            'revenue_growth': base_assumptions['revenue_growth'] + 0.02,
            'ebit_margin': base_assumptions['ebit_margin'] + 0.02,
            'wacc': max(base_assumptions['wacc'] - 0.01, 0.01)
        }

        bear_iv, _, _, _ = run_dcf_model(financials, bear_assumptions)
        neutral_iv = base_iv
        bull_iv, _, _, _ = run_dcf_model(financials, bull_assumptions)

        plt.figure(figsize=(8, 6))
        scenario_vals = [bear_iv, neutral_iv, bull_iv]
        labels = ['Bear', 'Neutral', 'Bull']
        colors = ['#f28b82', '#e0e0e0', '#b7e1a1']
        bars = plt.bar(labels, scenario_vals, color=colors)
        plt.ylabel('Intrinsic Value per Share (USD)')
        plt.title('Intrinsic Value: Bear vs. Neutral vs. Bull Case')
        for bar, val in zip(bars, scenario_vals):
            plt.text(bar.get_x() + bar.get_width()/2, val,
                     f'${val:,.0f}', ha='center', va='bottom')
        plt.show()

        # 3. Heatmap: Intrinsic value at different WACC & terminal growth
        display(Markdown("### WACC vs Terminal Growth Heatmap"))
        # Define ranges around base assumptions
        wacc_range = np.linspace(base_assumptions['wacc'] - 0.02,
                                 base_assumptions['wacc'] + 0.02, 7)
        tg_range = np.linspace(base_assumptions['terminal_growth'] - 0.01,
                               base_assumptions['terminal_growth'] + 0.01, 5)
        # Keep values in reasonable bounds
        wacc_range = np.clip(wacc_range, 0.01, 0.25)
        tg_range = np.clip(tg_range, 0.0, 0.10)

        heat_data = []
        for g in tg_range:
            row = []
            for w in wacc_range:
                # Avoid invalid perpetuity combos
                if (base_assumptions['terminal_method'] == 'Perpetuity Growth') and (g >= w):
                    row.append(np.nan)
                else:
                    ass = {**base_assumptions, 'wacc': w, 'terminal_growth': g}
                    iv, _, _, _ = run_dcf_model(financials, ass)
                    row.append(iv)
            heat_data.append(row)

        heat_df = pd.DataFrame(
            heat_data,
            index=[f'{g:.2%}' for g in tg_range],
            columns=[f'{w:.2%}' for w in wacc_range]
        )

        plt.figure(figsize=(8, 6))
        sns.heatmap(
            heat_df,
            annot=True,
            fmt='.0f',
            cmap='viridis',
            cbar_kws={'label': 'Intrinsic Value per Share (USD)'}
        )
        plt.xlabel('WACC')
        plt.ylabel('Terminal Growth Rate')
        plt.title('Intrinsic Value Heatmap')
        plt.show()

        # 4. Sensitivity table to key assumptions (+1% changes)
        display(Markdown("### Sensitivity to Key Assumptions"))
        sens_rows = []

        def add_sens_row(label, param_key, delta):
            up_ass = {**base_assumptions,
                      param_key: base_assumptions[param_key] + delta}
            up_iv, _, _, _ = run_dcf_model(financials, up_ass)
            value_impact = up_iv - base_iv
            pct_impact = value_impact / base_iv if base_iv != 0 else np.nan
            sens_rows.append([label, up_iv, value_impact, pct_impact])

        add_sens_row('+1% Revenue Growth', 'revenue_growth', 0.01)
        add_sens_row('+1% EBIT Margin', 'ebit_margin', 0.01)
        add_sens_row('+1% WACC', 'wacc', 0.01)

        sens_df = pd.DataFrame(
            sens_rows,
            columns=['Assumption Change', 'New Intrinsic Value', 'Value Impact', '% Impact']
        )
        display(
            sens_df.style.format({
                'New Intrinsic Value': '${:,.2f}',
                'Value Impact': '${:,.2f}',
                '% Impact': '{:.2%}'
            })
        )

        # 5. Monte Carlo Simulation with MC mean line
        display(Markdown("### Monte Carlo Simulation"))
        display(Markdown("_Note: A sample of 20,000 iterations is used._"))

        num_simulations = 20000
        results = [
            run_dcf_model(
                financials,
                {
                    **base_assumptions,
                    'revenue_growth': np.random.normal(
                        base_assumptions['revenue_growth'], 0.02
                    ),
                    'wacc': np.random.normal(
                        base_assumptions['wacc'], 0.01
                    )
                }
            )[0]
            for _ in range(num_simulations)
            if base_assumptions['wacc'] > base_assumptions['terminal_growth']
        ]

        if results:
            mc_mean = float(np.mean(results))
        else:
            mc_mean = float('nan')

        plt.figure(figsize=(10, 6))
        sns.histplot(results, kde=True, bins=50)
        plt.axvline(
            base_iv,
            color='red',
            linestyle='--',
            label=f'Base Case: ${base_iv:,.2f}'
        )
        plt.axvline(
            current_price,
            color='black',
            linestyle='--',
            label=f'Market Price: ${current_price:,.2f}'
        )
        if not np.isnan(mc_mean):
            plt.axvline(
                mc_mean,
                color='blue',
                linestyle='--',
                label=f'MC Mean: ${mc_mean:,.2f}'
            )
        plt.title('Distribution of Intrinsic Values (Monte Carlo)')
        plt.xlabel('Intrinsic Value')
        plt.ylabel('Frequency')
        plt.legend()
        plt.show()

        if not np.isnan(mc_mean):
            mc_text = f"Monte Carlo mean intrinsic value: ${mc_mean:,.2f}"
        else:
            mc_text = "Monte Carlo mean intrinsic value: N/A"

        display(Markdown(f"**{mc_text}**"))

        # 6. Final Valuation Report (clean text style)
        status = "Undervalued" if base_iv > current_price else "Overvalued"
        upside = (base_iv / current_price) - 1 if current_price != 0 else np.nan

        report_md = f"""
### Valuation Report: {ticker_input.value.upper()}

**Final Intrinsic Value:** ${base_iv:,.2f} | **Current Price:** ${current_price:,.2f}
**Upside/Downside:** {upside:.2%} ({status})

---

*Key Assumptions:* WACC: {base_assumptions['wacc']:.2%}, \
Rev Growth: {base_assumptions['revenue_growth']:.2%}, \
Term. Growth: {base_assumptions['terminal_growth']:.2%}

*Monte Carlo Mean Intrinsic Value:* {"N/A" if np.isnan(mc_mean) else f"${mc_mean:,.2f}"}
"""
        display(Markdown(report_md))


def get_dcf_inputs():
    return {
        'revenue_growth': growth_slider.value,
        'ebit_margin': margin_slider.value,
        'wacc': wacc_slider.value,
        'tax_rate': tax_rate_slider.value,
        'terminal_method': terminal_method_choice.value,
        'terminal_growth': terminal_growth_slider.value,
        'exit_multiple': exit_multiple_slider.value,
        'growth_model': growth_model_choice.value,
        'high_growth_years': high_growth_years_slider.value
    }


# Run the setup function to load all libraries
setup_environment()


Environment setup complete. All libraries are imported and ready.


In [6]:
# User Inputs and Interactive Controls (Final Advanced Version)

# --- Global variables and initial widget definitions ---
price_data = pd.Series(dtype=np.float64); financials = pd.DataFrame(); company_info = {}
style = {'description_width': '150px'}

# --- STEP 1: Ticker Input ---
ticker_input = widgets.Text(value='MSFT', placeholder='Enter Stock Ticker', description="Company Ticker:", style={'description_width': 'initial'})
fetch_button = widgets.Button(description="Fetch & Display Company Data")
reset_button = widgets.Button(description="Reset Assumptions", button_style='warning')
data_output = widgets.Output()

# --- STEP 2: DCF Assumptions ---
# Growth Model Selection
growth_model_choice = widgets.RadioButtons(options=['Single-Stage (5yr)', 'Multi-Stage (10yr)'], description='Forecast Model:', style=style)
high_growth_years_slider = widgets.IntSlider(value=5, min=1, max=9, step=1, description='High Growth Years:')
fade_years_display = widgets.HTML(value="<p><i>Fade Period: 5 years</i></p>")
multi_stage_box = widgets.HBox([high_growth_years_slider, fade_years_display], layout={'visibility': 'hidden'})
# WACC Components
risk_free_rate_choice = widgets.RadioButtons(options=['Manual', 'Auto'], description='Risk-Free Rate:', style=style)
risk_free_rate_slider = widgets.FloatSlider(value=0.042, min=0.01, max=0.07, step=0.001, readout_format='.2%')
market_premium_choice = widgets.RadioButtons(options=['Manual', 'Auto (6.0%)'], description='Market Premium:', style=style, value='Auto (6.0%)')
market_premium_slider = widgets.FloatSlider(value=0.06, min=0.03, max=0.10, step=0.005, readout_format='.1%')
cost_of_debt_choice = widgets.RadioButtons(options=['Manual', 'Auto'], description='Cost of Debt:', style=style)
cost_of_debt_slider = widgets.FloatSlider(value=0.055, min=0.02, max=0.10, step=0.005, readout_format='.1%')
tax_rate_choice = widgets.RadioButtons(options=['Manual', 'Auto'], description='Tax Rate:', style=style)
tax_rate_slider = widgets.FloatSlider(value=0.21, min=0.10, max=0.35, step=0.005, readout_format='.1%')
# Core Assumptions
growth_slider_label = widgets.HTML(value="<p><b>Growth Rate (Yrs 1-5)</b></p>")
growth_slider = widgets.FloatSlider(value=0.10, min=-0.1, max=0.3, step=0.005, readout_format='.1%')
growth_box = widgets.HBox([widgets.RadioButtons(options=['Manual', 'Auto'], description='Revenue Growth:', style=style), growth_slider])
margin_box = widgets.HBox([widgets.RadioButtons(options=['Manual', 'Auto'], description='EBIT Margin:', style=style), widgets.FloatSlider(value=0.25, min=0.0, max=0.6, step=0.005, readout_format='.1%')])
# WACC and Terminal Value
wacc_label = widgets.HTML(value="<p><i>(Note: WACC must be > Terminal Growth)</i></p>")
wacc_box = widgets.HBox([widgets.RadioButtons(options=['Manual', 'Auto (CAPM)'], description='WACC:', style=style), widgets.FloatSlider(value=0.085, min=0.03, max=0.15, step=0.005, readout_format='.1%'), wacc_label])
terminal_method_choice = widgets.RadioButtons(options=['Perpetuity Growth', 'Exit Multiple'], description='Terminal Method:', style=style)
terminal_growth_slider = widgets.FloatSlider(value=0.025, min=0.0, max=0.05, step=0.005, description='Terminal Growth:', readout_format='.1%', style=style)
exit_multiple_slider = widgets.FloatSlider(value=12.0, min=3.0, max=25.0, step=0.5, description='Exit Multiple (EV/EBIT):', style=style)

# --- STEP 3 & 4: Action Buttons ---
run_dcf_button = widgets.Button(description="Run DCF Valuation"); dcf_output = widgets.Output()
analysis_button = widgets.Button(description="Run Full Analysis & Report"); analysis_output = widgets.Output()

# --- WIDGET ALIASES (for easier access in functions) ---
growth_choice, growth_slider = growth_box.children
margin_choice, margin_slider = margin_box.children
wacc_choice, wacc_slider, _ = wacc_box.children

# --- Interactivity Logic ---
def on_fetch_button_clicked(b):
    fetch_and_display_data(ticker_input.value, data_output)
    if not financials.empty:
        # Update labels with auto-calculated values
        growth_choice.options = ['Manual', f"Auto ({financials['Total Revenue'].pct_change().tail(2).mean():.2%})"]
        margin_choice.options = ['Manual', f"Auto ({(financials['EBIT']/financials['Total Revenue'].replace(0,np.nan)).tail(2).mean():.2%})"]
        risk_free_rate_choice.options = ['Manual', f"Auto ({company_info.get('risk_free_rate_auto',0):.2%})"]
        tax_rate_choice.options = ['Manual', f"Auto ({company_info.get('tax_rate_auto',0):.2%})"]
        cost_of_debt_choice.options = ['Manual', f"Auto ({company_info.get('cost_of_debt_auto',0):.2%})"]
        update_auto_wacc_label()

def update_auto_wacc_label(*args):
    if not financials.empty and company_info:
        rfr = risk_free_rate_slider.value if 'Manual' in risk_free_rate_choice.value else company_info.get('risk_free_rate_auto', 0.04)
        mrp = market_premium_slider.value if 'Manual' in market_premium_choice.value else 0.06
        cod = cost_of_debt_slider.value if 'Manual' in cost_of_debt_choice.value else company_info.get('cost_of_debt_auto', 0.05)
        tax = tax_rate_slider.value if 'Manual' in tax_rate_choice.value else company_info.get('tax_rate_auto', 0.21)
        beta = company_info.get('beta', 1.0) or 1.0; market_cap = company_info.get('marketCap', 0)
        total_debt = financials.iloc[-1].get('Total Debt', 0)
        total_value = market_cap + total_debt
        equity_weight = market_cap/total_value if total_value>0 else 1.0; debt_weight = total_debt/total_value if total_value>0 else 0.0
        cost_of_equity = rfr + beta * mrp
        auto_wacc = (cost_of_equity * equity_weight) + (cod * debt_weight * (1 - tax))
        company_info['wacc_auto'] = auto_wacc
        wacc_choice.options = ['Manual', f'Auto (CAPM: {auto_wacc:.2%})']
        if 'Auto' in wacc_choice.value: wacc_slider.value = auto_wacc

def on_choice_change(choice, slider, auto_value_func):
    if 'Auto' in choice.new and not financials.empty:
        try: slider.value = auto_value_func(); slider.disabled = True
        except: slider.disabled = False
    else: slider.disabled = False

def toggle_growth_model(change):
    is_multi_stage = change.new == 'Multi-Stage (10yr)'
    multi_stage_box.layout.visibility = 'visible' if is_multi_stage else 'hidden'
    growth_slider_label.value = "<p><b>High Growth Rate (Yrs 1-{})</b></p>".format(high_growth_years_slider.value) if is_multi_stage else "<p><b>Growth Rate (Yrs 1-5)</b></p>"

def update_fade_years(change):
    fade_years = 10 - change.new
    fade_years_display.value = f"<p><i>Fade Period: {fade_years} years</i></p>"
    growth_slider_label.value = "<p><b>High Growth Rate (Yrs 1-{})</b></p>".format(change.new)

def reset_defaults(b):
    # Reset all widgets to their initial states
    risk_free_rate_choice.value = 'Manual'; risk_free_rate_slider.value = 0.042
    market_premium_choice.value = 'Auto (6.0%)'; market_premium_slider.value = 0.06
    cost_of_debt_choice.value = 'Manual'; cost_of_debt_slider.value = 0.055
    tax_rate_choice.value = 'Manual'; tax_rate_slider.value = 0.21
    growth_choice.value = 'Manual'; growth_slider.value = 0.10
    margin_choice.value = 'Manual'; margin_slider.value = 0.25
    wacc_choice.value = 'Manual'; wacc_slider.value = 0.085
    terminal_method_choice.value = 'Perpetuity Growth'
    growth_model_choice.value = 'Single-Stage (5yr)'
    high_growth_years_slider.value = 5

# --- Link all widgets to functions ---
fetch_button.on_click(on_fetch_button_clicked)
reset_button.on_click(reset_defaults)
run_dcf_button.on_click(lambda b: run_valuation_display(dcf_output))
analysis_button.on_click(lambda b: run_full_analysis(analysis_output))
growth_model_choice.observe(toggle_growth_model, names='value')
high_growth_years_slider.observe(update_fade_years, names='value')
growth_choice.observe(lambda c: on_choice_change(c, growth_slider, lambda: financials['Total Revenue'].pct_change().tail(2).mean()), names='value')
margin_choice.observe(lambda c: on_choice_change(c, margin_slider, lambda: (financials['EBIT']/financials['Total Revenue'].replace(0,np.nan)).tail(2).mean()), names='value')
risk_free_rate_choice.observe(lambda c: on_choice_change(c, risk_free_rate_slider, lambda: company_info.get('risk_free_rate_auto')), names='value')
market_premium_choice.observe(lambda c: on_choice_change(c, market_premium_slider, lambda: 0.06), names='value')
cost_of_debt_choice.observe(lambda c: on_choice_change(c, cost_of_debt_slider, lambda: company_info.get('cost_of_debt_auto')), names='value')
tax_rate_choice.observe(lambda c: on_choice_change(c, tax_rate_slider, lambda: company_info.get('tax_rate_auto')), names='value')
wacc_choice.observe(lambda c: on_choice_change(c, wacc_slider, lambda: company_info.get('wacc_auto', 0.085)), names='value')
terminal_method_choice.observe(lambda c: (setattr(terminal_growth_slider.layout, 'display', 'flex' if c.new=='Perpetuity Growth' else 'none'), setattr(exit_multiple_slider.layout, 'display', 'none' if c.new=='Perpetuity Growth' else 'flex')), names='value')
for item in [risk_free_rate_slider, market_premium_slider, cost_of_debt_slider, tax_rate_slider, risk_free_rate_choice, market_premium_choice, cost_of_debt_choice, tax_rate_choice]:
    item.observe(update_auto_wacc_label, names='value')

# --- Display Final Layout ---
exit_multiple_slider.layout.display = 'none'
display(Markdown("## Step 1: Fetch Company Data"))
display(widgets.HBox([ticker_input, fetch_button, reset_button]), data_output)
display(Markdown("## Step 2: Configure DCF Assumptions"))
display(widgets.VBox([
    widgets.HBox([risk_free_rate_choice, risk_free_rate_slider, market_premium_choice, market_premium_slider]),
    widgets.HBox([cost_of_debt_choice, cost_of_debt_slider, tax_rate_choice, tax_rate_slider]),
    widgets.HBox([growth_model_choice, multi_stage_box]),
    widgets.HBox([growth_slider_label, growth_box]),
    margin_box, wacc_box,
    widgets.HBox([terminal_method_choice, terminal_growth_slider, exit_multiple_slider])
]))
display(Markdown("## Step 3: Run the Valuation Model"))
display(run_dcf_button, dcf_output)
display(Markdown("## Step 4: Run Deeper Analysis & See Final Report"))
display(analysis_button, analysis_output)

## Step 1: Fetch Company Data

Output()

## Step 2: Configure DCF Assumptions

## Step 3: Run the Valuation Model

Button(description='Run DCF Valuation', style=ButtonStyle())

Output()

## Step 4: Run Deeper Analysis & See Final Report

Button(description='Run Full Analysis & Report', style=ButtonStyle())

Output()